# Module 12 Solutions: Differential Equations for AI

This notebook contains complete solutions and working Python code for the Module 12 exercises.

### 1. Analytical Solution of Linear System
For $\frac{d\mathbf{y}}{dt} = \begin{bmatrix} -1 & 0 \\ 0 & -2 \end{bmatrix} \mathbf{y}$ with $\mathbf{y}(0) = [2, -3]^T$:
- $y_1(t) = 2 e^{-t}$
- $y_2(t) = -3 e^{-2t}$

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp

def sys_ode(t, y):
    return [-y[0], -2*y[1]]

sol = solve_ivp(sys_ode, (0, 2), [2, -3], t_eval=[2.0])
print("Numerical Solution at t=2:", sol.y[:, 0])
print("Analytical Solution at t=2:", [2*np.exp(-2.0), -3*np.exp(-4.0)])

### 2. Implement Euler's Method from Scratch

In [ ]:
def euler_ode(h):
    t_vals = np.arange(0, 2+h, h)
    y = 1.0
    for _ in t_vals[:-1]:
        y = y + h * y
    return y

y_01 = euler_ode(0.1)
y_001 = euler_ode(0.01)
true_val = np.exp(2.0)
print(f"True: {true_val:.4f}, h=0.1: {y_01:.4f} (Err: {abs(y_01-true_val):.4f}), h=0.01: {y_001:.4f} (Err: {abs(y_001-true_val):.4f})")

### 3. Linearization and Equilibrium
For $f(x, y) = y - 1 = 0 \implies y^* = 1$.
For $g(x, y) = x^2 - y = 0 \implies x^2 - 1 = 0 \implies x^* = \pm 1$.
Equilibrium points: $(1, 1)$ and $(-1, 1)$.

Jacobian: $J(x, y) = \begin{bmatrix} 0 & 1 \\ 2x & -1 \end{bmatrix}$.
- At $(1, 1)$: $J = \begin{bmatrix} 0 & 1 \\ 2 & -1 \end{bmatrix}$. Eigenvalues: $\det(J-\lambda I) = \lambda^2 + \lambda - 2 = (\lambda+2)(\lambda-1) = 0 \implies \lambda = 1, -2$. Bounded positive real part: **Saddle/Unstable**.
- At $(-1, 1)$: $J = \begin{bmatrix} 0 & 1 \\ -2 & -1 \end{bmatrix}$. Eigenvalues: $\lambda^2 + \lambda + 2 = 0 \implies \lambda = \frac{-1 \pm i\sqrt{7}}{2}$. Negative real part: **Stable Spiral**.

In [ ]:
# Stability verified analytically.

### 4. RK4 Implementation

In [ ]:
def f_pendulum(t, state):
    theta, omega = state
    return [omega, -np.sin(theta)]

def rk4_step(f, t, y, h):
    k1 = np.array(f(t, y))
    k2 = np.array(f(t + h/2, y + h*k1/2))
    k3 = np.array(f(t + h/2, y + h*k2/2))
    k4 = np.array(f(t + h, y + h*k3))
    return y + (h/6.0) * (k1 + 2*k2 + 2*k3 + k4)

# Simulate pendulum trajectory
h = 0.05
t_eval = np.arange(0, 10, h)
state = np.array([2.0, 0.0])  # release from 2 radians
path = [state.copy()]

for t in t_eval[:-1]:
    state = rk4_step(f_pendulum, t, state, h)
    path.append(state.copy())
path = np.array(path)
print("RK4 Simulation final state (theta, omega):", path[-1])

### 5. Phase Portrait of Saddle Node

In [ ]:
import matplotlib.pyplot as plt

Y, X = np.mgrid[-5:5:100j, -5:5:100j]
U = X - Y
V = X**2 - 4

plt.figure(figsize=(6, 6))
plt.streamplot(X, Y, U, V, color=np.sqrt(U**2 + V**2), cmap='autumn')
plt.title("Phase Portrait")
plt.xlabel("x")
plt.ylabel("y")
plt.grid(True)
plt.show()

### 6. Matrix Exponential Calculation
For $A = \begin{bmatrix} 0 & 1 \\ -1 & 0 \end{bmatrix}$, notice that $A^2 = -I$, $A^3 = -A$, $A^4 = I$, etc. 
Using the Taylor expansion:
$$e^{At} = I + At + \frac{A^2 t^2}{2!} + \frac{A^3 t^3}{3!} + \dots$$
$$= \left( 1 - \frac{t^2}{2!} + \dots \right) I + \left( t - \frac{t^3}{3!} + \dots \right) A$$
$$= \cos(t) I + \sin(t) A = \begin{bmatrix} \cos(t) & \sin(t) \\ -\sin(t) & \cos(t) \end{bmatrix}$$

In [ ]:
import scipy.linalg as la
A = np.array([[0.0, 1.0], [-1.0, 0.0]])
print("Matrix exponential at t=1:\n", la.expm(A * 1.0))
print("Analytical formula output:\n", np.array([[np.cos(1), np.sin(1)], [-np.sin(1), np.cos(1)]]))

### 7. Adjoint Equation Derivation
Let $L = g(y(T))$ where $y(t)$ satisfies $\frac{dy}{dt} = f(y(t))$.
To compute $\frac{\partial L}{\partial y(t)}$, we define the adjoint $a(t) = \frac{\partial L}{\partial y(t)}$.
Using the chain rule, the change in $a(t)$ over an infinitesimal step $dt$ is:
$$a(t) = a(t+dt) \frac{\partial y(t+dt)}{\partial y(t)}$$
Using the Euler step $y(t+dt) \approx y(t) + dt f(y(t))$, we get:
$$\frac{\partial y(t+dt)}{\partial y(t)} = 1 + dt f'(y(t))$$
Substituting:
$$a(t) = a(t+dt) (1 + dt f'(y(t)))$$
$$a(t) - a(t+dt) = dt a(t+dt) f'(y(t))$$
Divide by $dt$ and take $dt \to 0$:
$$\frac{da(t)}{dt} = -a(t) f'(y(t))$$
The boundary condition at the end is by definition $a(T) = \frac{\partial L}{\partial y(T)} = g'(y(T))$.

In [ ]:
# Theoretical proof.

### 8. Forward vs Reverse SDE
For the OU process $dx_t = -\theta x_t dt + \sigma dw_t$:
The transition probability density $p(x_t|x_0)$ is Gaussian:
$$p(x_t|x_0) = \mathcal{N}\left( x_t \;\Big|\; x_0 e^{-\theta t}, \; \frac{\sigma^2}{2\theta} (1 - e^{-2\theta t}) \right)$$

In [ ]:
# Theoretical equation verified.

### 9. Implement a Toy Neural ODE from Scratch

In [ ]:
# Simple NumPy-based parameter fitting of Neural ODE dy/dt = theta * y
# We want to fit y(t=1) = 4 starting from y(0) = 2.
theta = 0.1  # initial guess
lr = 0.1
target = 4.0

for epoch in range(20):
    # Forward pass: solve ODE
    # y(t) = y(0) * e^(theta * t)
    y_T = 2.0 * np.exp(theta * 1.0)
    loss = (y_T - target)**2
    
    # Gradient dL/dtheta = dL/dy_T * dy_T/dtheta
    dl_dy = 2 * (y_T - target)
    dy_dtheta = 2.0 * np.exp(theta) * 1.0
    grad = dl_dy * dy_dtheta
    
    theta -= lr * grad
    if epoch % 5 == 0:
        print(f"Epoch {epoch}: theta = {theta:.4f}, Loss = {loss:.4f}, Pred = {y_T:.4f}")

### 10. Score-Based Generative Model: Reverse SDE

In [ ]:
# Forward noising: dx = -x dt + dw
# Reverse denoising: dx = (-x - score) dt + dw
# Mock score: score = -x / (1 - exp(-2t))

N = 100
t_steps = np.linspace(1e-3, 1.0, N)
dt = 1.0 / N

# Start with noise
x = np.random.normal(0, 1)

# Simulate backward in time
for t in reversed(t_steps):
    score = -x / (1.0 - np.exp(-2*t))
    drift = -x - 1.0 * score
    dw = np.random.normal(0, np.sqrt(dt))
    # Backward step
    x = x - drift * dt + 1.0 * dw
print("Denoised/reconstructed value:", x)

### 11. Physics-Informed Neural Network (PINN) Loss Formulation

In [ ]:
# PyTorch-like pseudo-code implementing wave equation loss: u_tt - c^2 u_xx = 0
def pinn_wave_loss(model, t, x, c=1.0):
    # Assumes t, x are torch.Tensors with requires_grad=True
    # u = model(t, x)
    # u_t = torch.autograd.grad(u, t, create_graph=True)[0]
    # u_tt = torch.autograd.grad(u_t, t, create_graph=True)[0]
    # u_x = torch.autograd.grad(u, x, create_graph=True)[0]
    # u_xx = torch.autograd.grad(u_x, x, create_graph=True)[0]
    # pde_residual = u_tt - (c**2) * u_xx
    # return torch.mean(pde_residual**2)
    print("PINN loss equation: u_tt - (c^2) * u_xx")

### 12. Euler-Maruyama Method for SDEs

In [ ]:
N = 500
dt = 1.0 / N
num_trajectories = 50
x = np.ones((num_trajectories, N))

for t in range(1, N):
    dw = np.random.normal(0, np.sqrt(dt), num_trajectories)
    # dx = -x^3 dt + 0.5 dw
    x[:, t] = x[:, t-1] - (x[:, t-1]**3) * dt + 0.5 * dw

mean = np.mean(x, axis=0)
var = np.var(x, axis=0)
print(f"Final mean at T=1: {mean[-1]:.4f}, Final variance: {var[-1]:.4f}")

### 13. Heat Equation Numerical Solution

In [ ]:
nx_points = 50
dx = 1.0 / nx_points
dt = 0.001
alpha = 0.01

# FTCS coefficient check (must be <= 0.5 for stability)
assert alpha * dt / (dx**2) <= 0.5

u = np.zeros(nx_points)
u[20:30] = 1.0  # Initial temperature block

for t in range(100):
    u_next = u.copy()
    for i in range(1, nx_points-1):
        u_next[i] = u[i] + alpha * dt / (dx**2) * (u[i+1] - 2*u[i] + u[i-1])
    u = u_next

print("Temperature distribution after 100 steps:")
print("Max temp:", np.max(u))

### 14. Continuous-Time Normalizing Flows
In a Continuous Normalizing Flow (CNF), a probability density $p(\mathbf{h}(t))$ is evolved via a vector field $\frac{d\mathbf{h}}{dt} = f(\mathbf{h}(t), t, \theta)$.
The change in the log-probability density is given by the instantaneous change of variables formula:
$$\frac{d \log p(\mathbf{h}(t))}{dt} = -\text{tr}\left( \frac{\partial f(\mathbf{h}(t), t, \theta)}{\partial \mathbf{h}} \right)$$
Calculating the exact trace of the Jacobian is expensive ($O(D^2)$), so CNFs use **Hutchinson's trace estimator** to approximate the trace with $O(D)$ complexity using random vectors $\mathbf
{v}$:
$$\text{tr}(J) = \mathbb{E}_{\mathbf{v}} [\mathbf{v}^T J \mathbf{v}]$$

In [ ]:
# Theoretical explanation.

### 15. Chaotic Systems & Sensitivity to Initial Conditions

In [ ]:
def lorenz(t, state):
    x, y, z = state
    return [10*(y - x), x*(28 - z) - y, x*y - (8/3)*z]

t_span = (0, 30)
t_eval = np.linspace(0, 30, 3000)
sol1 = solve_ivp(lorenz, t_span, [1.0, 1.0, 1.0], t_eval=t_eval)
sol2 = solve_ivp(lorenz, t_span, [1.0, 1.0, 1.0001], t_eval=t_eval)

diff = np.linalg.norm(sol1.y - sol2.y, axis=0)

plt.figure(figsize=(8, 4))
plt.plot(t_eval, diff, 'r-')
plt.title("Sensitivity to Initial Conditions in the Lorenz System")
plt.xlabel("Time")
plt.ylabel("Distance between trajectories")
plt.grid(True)
plt.show()